In [14]:
from mpmath import coulombf, coulombg
import numpy as np
import mpmath
import numba
from scipy.integrate import simpson
from scipy.linalg import eigh
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar
from scipy.special import genlaguerre, gamma
from scipy.integrate import quad
from scipy.constants import hbar
# 设置高精度计算（50位小数）
mpmath.mp.dps = 50
# # 计算 F 和 G
# F_values = [mpmath.coulombf(l, eta, rho) for rho in rho_values]
# G_values = [mpmath.coulombg(l, eta, rho) for rho in rho_values]

In [15]:
e2=1.43997 ; hbarc=197.3269718 ; amu=931.49432
z1=86 ; m1=211*amu+-8.755330
z2=2 ; m2=4*amu+2.42491587
Q=8.8624
mu=m1*m2/(m1+m2)
v0=162.3 ; a0=0.4 ; r0=7.660; l=5; P0=0.03
rc=r0
k=np.sqrt(2*mu*Q/(hbarc**2))
eta= z1 * z2 * e2 * mu / (hbarc**2 * k)
# z1=1 ; m1=1.0078*amu
# z2=0 ; m2=1.0087*amu
# l=0
# mu=m1*m2/(m1+m2)

In [16]:
#potential function
@numba.njit
def wspot(r,v0,a0,r0):
    return -v0*(1+np.cosh(r0/a0))/(np.cosh(r/a0)+np.cosh(r0/a0))

@numba.njit
def vc(r,z1,z2,rc):
    return np.where(
        r < rc,
        z1 * z2 * e2 * (3 - r**2 / rc**2) /(2 * rc),
        z1 * z2 * e2 / r
    )
    
@numba.njit
def vpot(r,v0,a0,r0,z1,z2,rc,l):
    return wspot(r,v0,a0,r0)+vc(r,z1,z2,rc)

#coulomb function
def F_L(r, k, eta, L):
    """库仑函数 F_L(η, kr)"""
    return coulombf(L, eta, k * r)

def G_L(r, k, eta, L):
    """库仑函数 G_L(η, kr)"""
    return coulombg(L, eta, k * r)

In [17]:
def isotropic_ho_radial(n, l, r, alpha=1.0):
    # 计算归一化常数
    norm = np.sqrt(2 * alpha**(3/2) * gamma(n+1) / gamma(n + l + 3/2))
    
    # 计算径向部分
    xi = alpha * r**2
    laguerre = genlaguerre(n, l + 0.5)(xi)
    
    # 组合所有部分
    radial_wave = norm * (alpha**0.5 * r)**l * np.exp(-xi/2) * laguerre
    
    return radial_wave

# r*R_nl(r)
def phi_basis(n, l, r, alpha=1.0):
    
    return r * isotropic_ho_radial(n, l, r, alpha)

In [18]:
h=0.01
r = np.arange(1e-11, 40, h)

In [19]:
# 计算FL和VV,FL是coulomb函数的离散值,VV是势函数的离散值,被用于进行DWBA的积分步骤
FL = np.array([F_L(ri, k, eta, l) for ri in r], dtype=np.complex128)

In [20]:
def phi_basis_second_derivative(n, l, r, alpha=1.0, dr=h):
    """
    计算φ_{nl}(r)对r的二阶导数
    内部点使用五点中心差分法
    边界点仅使用phi数组中的值计算
    """
    phi = phi_basis(n, l, r, alpha)
    
    d2phi = np.zeros_like(phi)
    
    # 内部点使用五点中心差分法
    d2phi[2:-2] = (-phi[4:] + 16*phi[3:-1] - 30*phi[2:-2] 
                   + 16*phi[1:-3] - phi[:-4]) / (12*dr**2)
    
    # 边界点简单使用三点差分法（仅用phi数组）
    d2phi[0] = (phi[0] - 2*phi[1] + phi[2]) / (dr**2)  
    d2phi[1] = (phi[0] - 2*phi[1] + phi[2]) / (dr**2)  
    
    d2phi[-1] = (phi[-3] - 2*phi[-2] + phi[-1]) / (dr**2)  
    d2phi[-2] = (phi[-3] - 2*phi[-2] + phi[-1]) / (dr**2)  
    
    return d2phi

In [21]:
ndim = 50
H = np.zeros((ndim, ndim), dtype=np.complex128)


In [ ]:
def Model(m1,m2,z1,z2,l,rc,r0,v0,a0,P0):
    for i in range(ndim):
        for j in range(ndim):
            phi_i = phi_basis(i, l, r, alpha=0.5)
            phi_j = phi_basis(j, l, r, alpha=0.5)
            d2phi_j = phi_basis_second_derivative(j, l, r, alpha=0.5, dr=h)
            kinetic_term = (-hbarc**2/2/mu) * simpson(phi_i.conj() * d2phi_j , x=r)
            centrifugal_term = (hbarc**2/2/mu)*l * (l + 1) * simpson(phi_i.conj() * phi_j / r**2 , x=r)
            vpot_term = simpson(phi_i.conj() * vpot(r,v0,a0,r0,z1,z2,rc,l)*phi_j,x=r)
            H[i, j]=kinetic_term+vpot_term+centrifugal_term

    # 计算特征值和特征向量
    eigenvalues, eigenvectors = eigh(H)
    # 找到最小正特征值及其对应的特征向量
    min_positive_eigenvalue = min([ev for ev in eigenvalues if ev > 0])
    min_index = np.where(eigenvalues == min_positive_eigenvalue)[0][0]
    eigenvector = eigenvectors[:, min_index]
    wf = np.zeros_like(r, dtype=np.complex128)
    for n in range(len(eigenvector)):  # 遍历所有量子数n
        phi = phi_basis(n, l, r, alpha=1.0)  # 固定l值，只变化n
        wf += eigenvector[n] * phi  # 线性组合

    # 计算FL和VV,FL是coulomb函数的离散值,VV是势函数的离散值,被用于进行DWBA的积分步骤
    VV=vpot(r,v0,a0,r0,z1,z2,rc,l)-z1*z2*e2/r

    # 计算积分结果
    result = simpson(y=VV*FL*wf, x=r)
    # 计算半衰期
    Gamma = P0 * abs(result)**2 * 4*mu/hbarc**2/k
    T_half = hbarc * np.log(2) / Gamma * 1e-23/3



    return min_positive_eigenvalue,T_half


In [ ]:
E, t_half=Model(m1,m2,z1,z2,l,rc,r0,v0,a0,P0)
print(E)
print(t_half)

C:\Users\16437\AppData\Local\Temp\ipykernel_61412\3924229657.py:7: DeprecationWarning: You are passing x=[1.000e-11 1.000e-02 2.000e-02 ... 3.997e+01 3.998e+01 3.999e+01] as a positional argument. Please change your invocation to use keyword arguments. From SciPy 1.14, passing these as positional arguments will result in an error.
  kinetic_term = (-hbarc**2/2/mu) * simpson(phi_i.conj() * d2phi_j , r)
C:\Users\16437\AppData\Local\Temp\ipykernel_61412\3924229657.py:8: DeprecationWarning: You are passing x=[1.000e-11 1.000e-02 2.000e-02 ... 3.997e+01 3.998e+01 3.999e+01] as a positional argument. Please change your invocation to use keyword arguments. From SciPy 1.14, passing these as positional arguments will result in an error.
  centrifugal_term = (hbarc**2/2/mu)*l * (l + 1) * simpson(phi_i.conj() * phi_j / r**2 , r)
C:\Users\16437\AppData\Local\Temp\ipykernel_61412\3924229657.py:9: DeprecationWarning: You are passing x=[1.000e-11 1.000e-02 2.000e-02 ... 3.997e+01 3.998e+01 3.999e+01]

8.538742611246093
0.7382035294655903


### Def prior and posterior

In [24]:
y=[8.8624,1.68e-3]  #Q,T_half
sigma_Q=2.3e-3      #MeV
sigma_t=0.01e-3     #s
sigma=[sigma_Q,sigma_t]
#a=[v0,a0,r0,P]
def log_prior(a):
    vv,aa,rr,pp=a
    R=1
    v0=162.3; a0=0.4; r0=7.660; P0=0.03
    
    sigma_v=3*R
    sigma_a=0.1*R
    sigma_r=0.1*R
    sigma_P=0.01*R

    prior_v=-0.5*(vv-v0)**2/sigma_v**2
    prior_a=-0.5*(aa-a0)**2/sigma_a**2
    prior_r=-0.5*(rr-r0)**2/sigma_r**2
    prior_P=-0.5*(pp-P0)**2/sigma_P**2

    return prior_v+prior_a+prior_r+prior_P

#a=[v0,a0,r0,P]
def log_posterior(a,y,sigma,m1,m2,z1,z2,l):
    log_prior_value=log_prior(a)
    q,t=Model(m1,m2,z1,z2,l,a[2],a[2],a[0],a[1],a[3])
    log_likelihood=-0.5*((q-y[0])**2/sigma[0]**2+(t-y[1])**2/sigma[1]**2)
    return log_prior_value+log_likelihood

In [25]:
# M=4
# nwalkers=2*M
# initial_pos = [v0, a0, r0, P0]
# pertubation_scale=[1, 0.05, 0.1, 0.001]
# a=np.array([initial_pos+np.random.normal(0,pertubation_scale,M) for _ in range(nwalkers)])
# print(a)

# import emcee
# import multiprocessing
# with multiprocessing.Pool() as pool:
#     sampler = emcee.EnsembleSampler(nwalkers, M, log_posterior, args=[y,sigma,z1,z2,l,Q], a=0.2, pool=pool)
#     state = sampler.run_mcmc(a, 200)
#     sampler.reset()
#     sampler.run_mcmc(state, 2000)

In [26]:
# import prettyplease

# samples = sampler.get_chain(flat=True)

# labels=["$V_0$","$a_0","$r_0$","$P_0$"]
# fig = prettyplease.corner(samples, labels=labels)
# plt.show()